# RoadSafe Chicago | Partie 02 : Compréhension des données

## Rappel du projet

**RoadSafe Chicago** étudie l'accidentologie routière à Chicago à partir des données ouvertes de la ville.

**Problématique** : *Comment évoluent la fréquence et la gravité des accidents routiers à Chicago, et quels facteurs sont associés à leur gravité ?*

La partie 01 a permis de récupérer les accidents de **2024** depuis l'API et de les sauvegarder dans `data/raw/crashes_2024.parquet`.

---

## Objectif de cette partie

**Comprendre les données avant de les modifier.** On ne nettoie pas une variable sans savoir ce qu'elle signifie, ni si une valeur manquante est normale ou problématique.

L'enjeu est de savoir précisément ce que représente une ligne, ce que contient chaque colonne et quelles variables serviront à répondre à la problématique.

## Contenu de cette partie

1. **Vue d'ensemble** : dimensions du jeu de données, types des colonnes, premières lignes.
2. **Lecture du dictionnaire de données** : signification de chaque variable, à partir de la documentation officielle de la ville.
3. **Identification des variables clés** :
   - la **clé** : `crash_record_id` ;
   - les dimensions **temporelles** : date, heure, jour, mois ;
   - les dimensions **spatiales** : latitude, longitude, localisation ;
   - le **contexte** : météo, luminosité, état de la route, vitesse affichée ;
   - la **gravité** : nombre de blessés, de décès, niveau de blessure le plus grave.
4. **Premiers constats** : valeurs manquantes, modalités « UNKNOWN » ou « NOT REPORTED », catégories rares, valeurs surprenantes.

## Résultat attendu

- Un **inventaire des variables** utiles au projet, avec leur rôle dans l'analyse.
- Une description claire de la **granularité** : 1 ligne = 1 accident.
- **Trois constats** sur la structure des données et leurs limites, qui guideront la partie suivante : **le nettoyage et le contrôle qualité**.

> **Question de clôture** : peut-on expliquer ce que représente une ligne, quelles variables répondent à la problématique, et quels problèmes de qualité empêchent encore une analyse fiable ?

## Librairies 

In [1]:
import pandas as pd
import numpy as np


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 1. Chargement des données

In [3]:
df = pd.read_parquet("../data/raw/crashes_2024.parquet" )



print ( f"Voici la taille de la base de données : ", df.shape )
display(df.head(3))

Voici la taille de la base de données :  (112056, 49)


,crash_record_id,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,report_type,crash_type,private_property_i,hit_and_run_i,damage,date_police_notified,prim_contributory_cause,sec_contributory_cause,street_no,street_direction,street_name,beat_of_occurrence,num_units,crash_month,most_severe_injury,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,idot_control_no,latitude,longitude,location,intersection_related_i,crash_date_est_i,photos_taken_i,statements_taken_i,work_zone_i,work_zone_type,workers_present_i,dooring_i,lane_cnt
0,5082d6e9f9c5c96660e7ec7fd416549e50d3130af3fc5c...,2024-06-27T13:45:00.000,10,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,PARKING LOT,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NOT ON SCENE (DESK REPORT),NO INJURY / DRIVE AWAY,N,Y,"OVER $1,500",2024-06-27T14:11:00.000,NOT APPLICABLE,NOT APPLICABLE,7023,S,WESTERN AVE,832,2,6,NO INDICATION OF INJURY,0,0,0,0,0,2,0,13,5,X003465244,41.7658087973,-87.6831267408,"{'type': 'Point', 'coordinates': ['-87.683127'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fd34aa4f12f00106b32134531fc6e805590dbe89fef502...,2024-06-22T18:07:00.000,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,ON SCENE,NO INJURY / DRIVE AWAY,NaN,NaN,"OVER $1,500",2024-06-22T18:51:00.000,UNABLE TO DETERMINE,NOT APPLICABLE,10615,S,SANGAMON ST,2232,2,6,NO INDICATION OF INJURY,0,0,0,0,0,1,0,18,7,X003459643,41.7009215428,-87.646149348,"{'type': 'Point', 'coordinates': ['-87.646149'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f0a652bf48a8c08c8a26cc981e39ed757cd41f8a771761...,2024-06-23T08:41:00.000,10,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,REAR TO REAR,PARKING LOT,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NOT ON SCENE (DESK REPORT),NO INJURY / DRIVE AWAY,Y,NaN,"OVER $1,500",2024-06-23T13:50:00.000,FAILING TO YIELD RIGHT-OF-WAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,939,W,NORTH AVE,1822,2,6,NO INDICATION OF INJURY,0,0,0,0,0,2,0,8,1,X003461336,41.9107384033,-87.6526288748,"{'type': 'Point', 'coordinates': ['-87.652629'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Faisons l'inventaire du dataset 

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112056 entries, 0 to 112055
Data columns (total 49 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   crash_record_id                112056 non-null  str   
 1   crash_date                     112056 non-null  str   
 2   posted_speed_limit             112056 non-null  str   
 3   traffic_control_device         112056 non-null  str   
 4   device_condition               112056 non-null  str   
 5   weather_condition              112056 non-null  str   
 6   lighting_condition             112056 non-null  str   
 7   first_crash_type               112056 non-null  str   
 8   trafficway_type                112056 non-null  str   
 9   alignment                      112056 non-null  str   
 10  roadway_surface_cond           112056 non-null  str   
 11  road_defect                    112056 non-null  str   
 12  report_type                    106985 non-null  str   


In [5]:
df.isna().sum()

crash_record_id                       0
crash_date                            0
posted_speed_limit                    0
traffic_control_device                0
device_condition                      0
weather_condition                     0
lighting_condition                    0
first_crash_type                      0
trafficway_type                       0
alignment                             0
roadway_surface_cond                  0
road_defect                           0
report_type                        5071
crash_type                            0
private_property_i               107506
hit_and_run_i                     76183
damage                                0
date_police_notified                  0
prim_contributory_cause               0
sec_contributory_cause                0
street_no                             0
street_direction                      0
street_name                           0
beat_of_occurrence                    0
num_units                             0


**Analyse.** Seules quelques colonnes ont des valeurs manquantes, et ce sont presque toujours des champs facultatifs (indicateurs Y/N, zone de travaux, numéro IDOT). Les 1 189 coordonnées manquantes (environ 1 %) seront signalées en partie 03 plutôt que supprimées.

In [6]:
cols  = df.columns

cols


Index(['crash_record_id', 'crash_date', 'posted_speed_limit',
       'traffic_control_device', 'device_condition', 'weather_condition',
       'lighting_condition', 'first_crash_type', 'trafficway_type',
       'alignment', 'roadway_surface_cond', 'road_defect', 'report_type',
       'crash_type', 'private_property_i', 'hit_and_run_i', 'damage',
       'date_police_notified', 'prim_contributory_cause',
       'sec_contributory_cause', 'street_no', 'street_direction',
       'street_name', 'beat_of_occurrence', 'num_units', 'crash_month',
       'most_severe_injury', 'injuries_total', 'injuries_fatal',
       'injuries_incapacitating', 'injuries_non_incapacitating',
       'injuries_reported_not_evident', 'injuries_no_indication',
       'injuries_unknown', 'crash_hour', 'crash_day_of_week',
       'idot_control_no', 'latitude', 'longitude', 'location',
       'intersection_related_i', 'crash_date_est_i', 'photos_taken_i',
       'statements_taken_i', 'work_zone_i', 'work_zone_type',
  

## 3. Classer les variables par rôle

Ce classement relie les colonnes aux quatre axes. Les noms sont ceux de l'API.

In [7]:
roles = {
    "CLÉ": ["crash_record_id"],

    "QUAND": ["crash_date", "crash_hour", "crash_day_of_week", "crash_month"],

    "OÙ": ["latitude", "longitude", "location",
           "street_no", "street_direction", "street_name", "beat_of_occurrence"],

    "CONTEXTE_ENVIRONNEMENT": ["weather_condition", "lighting_condition"],

    "CONTEXTE_ROUTE": ["posted_speed_limit", "traffic_control_device", "device_condition",
                       "trafficway_type", "alignment", "roadway_surface_cond", "road_defect",
                       "lane_cnt", "intersection_related_i", "private_property_i",
                       "work_zone_i", "work_zone_type", "workers_present_i"],

    "ACCIDENT": ["first_crash_type", "prim_contributory_cause", "sec_contributory_cause",
                 "num_units", "hit_and_run_i", "dooring_i"],

    "GRAVITÉ": ["crash_type", "damage", "most_severe_injury",
                "injuries_total", "injuries_fatal", "injuries_incapacitating",
                "injuries_non_incapacitating", "injuries_reported_not_evident",
                "injuries_no_indication", "injuries_unknown"],

    "ADMIN": ["report_type", "date_police_notified", "crash_date_est_i",
              "photos_taken_i", "statements_taken_i", "idot_control_no"],
}

In [8]:


# Contrôle : chaque colonne est classée une seule fois
classees = [c for cols in roles.values() for c in cols]
print("Colonnes classées :", len(classees), "/", df.shape[1])
print("Non classées :", set(df.columns) - set(classees))
print("Classées en double :", {c for c in classees if classees.count(c) > 1})

Colonnes classées : 49 / 49
Non classées : set()
Classées en double : set()


In [9]:


# Résumé par rôle
pd.DataFrame([
    {"rôle": r, "nb_colonnes": len(c), "taux_NA_moyen_%": round(df[c].isna().mean().mean() * 100, 1)}
    for r, c in roles.items()
])

,rôle,nb_colonnes,taux_NA_moyen_%
0,CLÉ,1,0.0
1,QUAND,4,0.0
2,OÙ,7,0.5
3,CONTEXTE_ENVIRONNEMENT,2,0.0
4,CONTEXTE_ROUTE,13,44.0
5,ACCIDENT,6,27.9
6,GRAVITÉ,10,0.2
7,ADMIN,6,48.9


**Analyse.** Les colonnes utiles à la problématique (QUAND, OÙ, CONTEXTE_ENVIRONNEMENT, GRAVITÉ) sont quasi complètes. Les taux élevés des rôles CONTEXTE_ROUTE (44 %) et ADMIN (49 %) viennent de colonnes secondaires renseignées seulement quand elles s'appliquent (zone de travaux, nombre de voies, numéro IDOT). Ils ne remettent pas en cause l'analyse.

### Variables catégorielles

In [10]:
cat_cols = ["weather_condition", "lighting_condition", "roadway_surface_cond",
            "most_severe_injury", "crash_type", "prim_contributory_cause"]

for col in cat_cols:
    print(f"\n ###{col} ({df[col].nunique()} modalités) ###")
    print(df[col].value_counts(normalize=True, dropna=False).mul(100).round(1).head(8))


 ###weather_condition (12 modalités) ###
weather_condition
CLEAR                    78.3
RAIN                      8.2
UNKNOWN                   7.9
CLOUDY/OVERCAST           2.8
SNOW                      1.9
FREEZING RAIN/DRIZZLE     0.3
OTHER                     0.3
FOG/SMOKE/HAZE            0.1
Name: proportion, dtype: float64

 ###lighting_condition (6 modalités) ###
lighting_condition
DAYLIGHT                  63.4
DARKNESS, LIGHTED ROAD    21.5
UNKNOWN                    6.3
DARKNESS                   4.5
DUSK                       2.7
DAWN                       1.7
Name: proportion, dtype: float64

 ###roadway_surface_cond (7 modalités) ###
roadway_surface_cond
DRY                72.7
UNKNOWN            13.1
WET                11.7
SNOW OR SLUSH       1.8
ICE                 0.4
OTHER               0.3
SAND, MUD, DIRT     0.0
Name: proportion, dtype: float64

 ###most_severe_injury (5 modalités) ###
most_severe_injury
NO INDICATION OF INJURY     83.3
NONINCAPACITATING INJURY   

**Analyse.** La météo est dominée par le temps clair (78 %) et la luminosité par le plein jour (63 %) : les conditions dégradées représentent des effectifs modestes, ce qui imposera de raisonner en **taux** accompagnés d'intervalles de confiance plutôt qu'en volumes.

La part de valeurs inconnues varie beaucoup selon la variable : 6 à 8 % pour la météo et la luminosité, 13 % pour la chaussée, et près de la moitié des causes principales une fois regroupées les modalités « UNABLE TO DETERMINE » et « NOT APPLICABLE ». Les causes déclarées devront donc être interprétées avec une grande prudence.

### Aperçu des variables numériques

Ici, la conversion des types est temporaire, uniquement pour explorer. La vraie conversion se fera dans la partie 03, et rien n'est sauvegardé à cette étape.

In [11]:
num_cols = ["posted_speed_limit", "injuries_total", "injuries_fatal", "num_units",
            "crash_hour", "latitude", "longitude"]
df[num_cols].apply(pd.to_numeric, errors="coerce").describe().round(2)

,posted_speed_limit,injuries_total,injuries_fatal,num_units,crash_hour,latitude,longitude
count,112056.00,111807.00,111807.00,112056.00,112056.00,110867.00,110867.00
mean,28.56,0.23,0.00,2.04,13.19,41.86,-87.67
std,5.53,0.62,0.04,0.45,5.60,0.34,0.70
min,0.00,0.00,0.00,1.00,0.00,0.00,-87.92
25%,30.00,0.00,0.00,2.00,9.00,41.79,-87.72
50%,30.00,0.00,0.00,2.00,14.00,41.88,-87.68
75%,30.00,0.00,0.00,2.00,17.00,41.93,-87.64
max,70.00,15.00,2.00,18.00,23.00,42.02,0.00


Les variables numériques sont globalement cohérentes, mais révèlent des valeurs « techniques » à corriger : des coordonnées à (0, 0) situées hors de Chicago, environ 1 % d’accidents non géolocalisés et des limites de vitesse à 0 mph. La gravité est par ailleurs très déséquilibrée : au moins 75 % des accidents ne font aucun blessé et les accidents mortels sont rares. L’analyse devra donc s’appuyer sur des proportions et des effectifs suffisants plutôt que sur des moyennes.

## 4. Vérification des dates

In [12]:
dates = pd.to_datetime(df["crash_date"], errors="coerce")
print("Période :", dates.min(), "=>", dates.max())
print("Accidents par mois :")
print(dates.dt.month.value_counts().sort_index())

Période : 2024-01-01 00:00:00 => 2024-12-31 23:47:00
Accidents par mois :
crash_date
1      8234
2      7800
3      8927
4      9254
5     10742
6     10136
7      9606
8      9924
9      9816
10     9891
11     8937
12     8789
Name: count, dtype: int64


**Analyse.** La période couvre bien l'année 2024 complète, du 1er janvier au 31 décembre. Les volumes mensuels vont de 7 800 accidents (février) à 10 742 (mai) : c'est un premier signe de saisonnalité, à confirmer en ramenant chaque mois à son nombre de jours (partie 04), puis par un test statistique (partie 07).

## Constats

1. **Structure** : 112 056 accidents en 2024, décrits par 49 colonnes. Une ligne correspond à un accident, identifié par la clé unique `crash_record_id`. Toutes les valeurs sont stockées en texte et devront être typées.

2. **Qualité des informations** : plusieurs valeurs inconnues ou aberrantes ont été repérées. Les modalités « UNKNOWN » représentent 7,9 % de la météo et 6,3 % de la luminosité. Environ 1 % des accidents ne sont pas géolocalisés, certains ont des coordonnées à (0, 0), et quelques limites de vitesse valent 0 mph.

3. **Gravité** : la distribution est très déséquilibrée, avec au moins 75 % d'accidents sans blessé et des accidents mortels rares. Les analyses de gravité reposeront donc principalement sur la part d'accidents avec blessés, et l'étude des décès nécessitera probablement l'extension à 2018-2025.